# Training: 46-Class EfficientNet-B0

Fine-tune an ImageNet-pretrained EfficientNet-B0 for 46-class fruit classification (fru92 subset).

In [1]:
import timm
import torch
import torch.nn as nn

NUM_CLASSES = 46
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)

print(f"Model: {model.default_cfg['architecture']}")
print(f"Classifier head: {model.get_classifier()}")

Model: efficientnet_b0
Classifier head: Linear(in_features=1280, out_features=46, bias=True)


In [2]:
from torchvision import datasets
from torch.utils.data import DataLoader

config = timm.data.resolve_model_data_config(model)
train_transform = timm.data.create_transform(**config, is_training=True)
val_transform = timm.data.create_transform(**config, is_training=False)

train_ds = datasets.ImageFolder("PrepData/Training", transform=train_transform)
val_ds = datasets.ImageFolder("PrepData/Validation", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"Classes: {train_ds.classes}")
print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")

Classes: ['Chinese_chestnut', 'Dangshan_Pear', 'Hami_melon', 'almond', 'annona_muricata', 'apple', 'apricot', 'artocarpus_heterophyllus', 'avocado', 'banana', 'bayberry', 'bergamot_pear', 'black_currant', 'black_grape', 'blood_orange', 'blueberry', 'breadfruit', 'candied_date', 'carambola', 'cashew_nut', 'cherry', 'cherry_tomato', 'citrus', 'coconut', 'crown_pear', 'dekopon', 'diospyros_lotus', 'durian', 'fig', 'flat_peach', 'gandaria', 'ginseng_fruit', 'golden_melon', 'grape', 'grape_white', 'grapefruit', 'green_apple', 'green_dates', 'guava', 'hawthorn', 'hazelnut', 'hickory', 'honey_dew_melon', 'housi_pear', 'juicy_peach', 'jujube']
Training samples: 23464, Validation samples: 5008


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print(f"Device: {device}")

Device: cuda


In [5]:
from tqdm.auto import tqdm

epochs = 25
for epoch in range(epochs):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation Phase
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / len(val_ds)
    print(f"Epoch {epoch+1}: Loss = {train_loss/len(train_loader):.4f}, Val Acc = {accuracy:.2f}%")

Epoch 1/25:   0%|          | 0/734 [00:00<?, ?it/s]

Epoch 1: Loss = 2.5838, Val Acc = 31.43%


Epoch 2/25:   0%|          | 0/734 [00:00<?, ?it/s]

Epoch 2: Loss = 2.4605, Val Acc = 37.74%


Epoch 3/25:   0%|          | 0/734 [00:00<?, ?it/s]

...

Epoch 23: Loss = 1.2662, Val Acc = 68.25%


Epoch 24/25:   0%|          | 0/734 [00:00<?, ?it/s]

Epoch 24: Loss = 1.2358, Val Acc = 69.71%


Epoch 25/25:   0%|          | 0/734 [00:00<?, ?it/s]

Epoch 25: Loss = 1.1995, Val Acc = 70.47%


In [7]:
torch.save(model.state_dict(), "fruit_classifier_46cls_b0.pth")